# Correlation plots

## Description
Plots per ROIs of correlations between amblyopic (AE) and fellow (FE) eye (patient) or left (LE) and right (RE) eye. <br>
We here compute and display these corretions for the pRF R2/x/y/ecc/size/CM. <br>
To do so we compute a quantile binning of the data per parameters and per ROI based on the AE data for each parameter. <br> 
For each bin we determine the median AE-RE and FE-LE median value and the 95 CI. <br>
We next compute a regression weighted by the median AE R2 of each bin. <br>
We plot this per ROI using size of dot function of the median AE R2 value. <br>

## Todo
- [x] get the data per subtasks
- [x] get the info about participant
- [x] add in the data the eye_val (right_eye) or (left_eye)
- [x] for patient add in the data the eye_type (fellow_eye) or (amblyopic_eye) 
- [x] for control add in the data the eye_type (right_eye) or (left_eye)
- [x] get average of parameters over a set of binned data (see ecc-size relationship)
- [x] get correlations over all values (see ecc-size relationship)
- [x] make the plots
- [x] clarify bin/rnages/qbin/threshold etc..
- [ ] make it a python code
- [ ] save the plot
- [ ] make it a group plot

In [ ]:
# Inputs
main_dir = '/home/mszinte/disks/meso_S/data'
project_dir = 'amblyo7T_prf'
subject = 'sub-17'
group = '327'

In [ ]:
# Debug
import ipdb
deb = ipdb.set_trace
# Stop warnings
import warnings
warnings.filterwarnings("ignore")

# General imports
import os
import sys
import pandas as pd
import scipy.stats

# Personal import
# '/home/mszinte/disks/meso_H/projects/pRF_analysis/amblyo7T_prf/postproc/prf/postfit/dev'
sys.path.append("{}/../../../../../analysis_code/utils".format(os.getcwd()))
from plot_utils import *
from settings_utils import load_settings
from maths_utils import make_prf_distribution_df, weighted_nan_median, weighted_nan_percentile, make_prf_barycentre_df, weighted_regression

# Load settings
base_dir = os.path.abspath(os.path.join(os.getcwd(), "../../../../../"))
settings_path = os.path.join(base_dir, project_dir, "settings.yml")
prf_settings_path = os.path.join(base_dir, project_dir, "prf-analysis.yml")
figure_settings_path = os.path.join(base_dir, project_dir, "figure-settings.yml")
settings = load_settings([settings_path, prf_settings_path, figure_settings_path])
analysis_info = settings[0]
formats = analysis_info['formats']
extensions = analysis_info['extensions']
rois_methods = analysis_info['rois_methods']
ecc_threshold = analysis_info['ecc_th']
size_threshold = analysis_info['size_th']
rsqr_threshold = analysis_info['rsqr_th']
amplitude_threshold = analysis_info['prf_amp_th']
stats_threshold = analysis_info['stats_th']
n_threshold = analysis_info['n_th']
subjects_to_group = analysis_info['subjects']
preproc_prep = analysis_info['preproc_prep']
filtering = analysis_info['filtering']
normalization = analysis_info['normalization']
avg_methods = analysis_info['avg_methods']

maps_names_pcm = analysis_info['maps_names_pcm']
maps_names_css_stats = analysis_info['maps_names_css_stats']
ecc_size_num_bins = analysis_info['ecc_size_num_bins']
ecc_pcm_num_bins = analysis_info['ecc_pcm_num_bins']
polar_angle_num_bins = analysis_info['polar_angle_num_bins']
ecc_pcm_max = analysis_info['ecc_pcm_max']
ecc_size_max = analysis_info['ecc_size_max']

# Load the participant tsv
participants_path = os.path.join(main_dir, project_dir, 'participants.tsv')
participants_df = pd.read_table(participants_path)
amblyopic_eyes = dict(zip(participants_df['participant_id'], participants_df['amblyopic_eye']))
amblyopic_eye = amblyopic_eyes[subject]

# Load subject group (patient or control)
groups = dict(zip(participants_df['participant_id'], participants_df['group']))
subject_group = groups[subject]

# to add to settings
prf_tasks_eyes_names = [['BarsBarsRingsWedgesRightEye', 'BarsBarsRingsWedgesLeftEye'],
                        ['BarsBarsRightEye', 'BarsBarsLeftEye'],
                        ['RingsWedgesRightEye', 'RingsWedgesLeftEye']]

prf_tasks_names = ['BarsBarsRingsWedges', 'BarsBars', 'RingsWedges']

corr_params = ['prf_rsq', 'prf_x', 'prf_y', 'prf_size', 'prf_ecc', 'pcm_median']

zscore_outlier_th = 2
corr_bin_number = 10

In [ ]:
for avg_method in avg_methods:
    if 'loo' in avg_method: rsq2use = 'prf_loo_rsq'
    else: rsq2use = 'prf_rsq'

    for format_, extension in zip(formats, extensions):
        rois_methods_format = rois_methods[format_]
        for rois_method_format in rois_methods_format:

            if rois_method_format == 'rois-drawn':
                rois = analysis_info[rois_method_format]
            elif rois_method_format == 'rois-group-mmp':
                rois = list(analysis_info[rois_method_format].keys())

            for prf_task_eyes_names in prf_tasks_eyes_names:

                data_task_eye = {}
                for prf_task_eye_name in prf_task_eyes_names:
        
                    print(f'Loading: {prf_task_eye_name} - {avg_method} - {format_}')
                
                    # Individual subject analysis
                    if 'group' not in subject:
                        
                        tsv_dir = '{}/{}/derivatives/pp_data/{}/{}/prf/tsv'.format(
                            main_dir, project_dir, subject, format_)
                        
                        # Exception if no data for one format (e.g template subject)
                        if not os.path.isdir(tsv_dir):
                            print(f"[SKIP] tsv_dir not found for format={format_}: {tsv_dir}")
                            continue
                        
                        fn_spec = "task-{}_{}_{}_{}_{}_{}".format(
                            prf_task_eye_name, preproc_prep, filtering,
                            normalization, avg_method, rois_method_format)
                        tsv_fn = '{}/{}_{}_prf-css-deriv.tsv'.format(
                            tsv_dir, subject, fn_spec)
                        data = pd.read_table(tsv_fn, sep="\t")
                       
                        # Keep a raw data df 
                        data_raw = data.copy()
                        
                        # Threshold data (replace by nan)
                        if stats_threshold == 0.05: stats_col = 'corr_pvalue_5pt'
                        elif stats_threshold == 0.01: stats_col = 'corr_pvalue_1pt'
                        data.loc[(data.amplitude < amplitude_threshold[0]) |
                                 (data.prf_ecc < ecc_threshold[0]) | (data.prf_ecc > ecc_threshold[1]) |
                                 (data.prf_size < size_threshold[0]) | (data.prf_size > size_threshold[1]) |
                                 (data.prf_n < n_threshold[0]) | (data.prf_n > n_threshold[1]) | 
                                 (data[rsq2use] < rsqr_threshold) |
                                 (data[stats_col] > stats_threshold)] = np.nan
                        
                        data = data.dropna()

                        # Get eye_val
                        if 'RightEye' in prf_task_eye_name:
                            eye_val = 'right eye'
                        elif 'LeftEye' in prf_task_eye_name:
                            eye_val = 'left eye'
                        data['eye_val'] = eye_val
    
                        # Get eye_type
                        if subject_group == 'patient':
                            amblyopic_side = amblyopic_eye[0]
                            if eye_val == 'right eye' and amblyopic_side == 'R': 
                                eye_type = 'amblyopic eye'
                            elif eye_val == 'right eye' and amblyopic_side == 'L': 
                                eye_type = 'fellow eye'
                            elif eye_val == 'left eye' and amblyopic_side == 'L': 
                                eye_type = 'amblyopic eye'
                            elif eye_val == 'left eye' and amblyopic_side == 'R': 
                                eye_type = 'fellow eye'
                        elif subject_group == 'control':
                            eye_type = eye_val
                        data['eye_type'] = eye_type

                        # Get suffixes BEFORE merging, based on eye_type
                        if len(data_task_eye) == 0:
                            data_task_eye['first'] = data
                            first_eye_type = eye_type
                        else:
                            second_eye_type = eye_type
                            
                            # Set suffixes based on eye_type order
                            if first_eye_type in ['amblyopic eye', 'right eye']:
                                suffix1, suffix2 = '_AE-RE', '_FE-LE'
                            else:
                                suffix1, suffix2 = '_FE-LE', '_AE-RE'
                                data_task_eye['first'], data = data, data_task_eye['first']
                            
                            data = pd.merge(data_task_eye['first'], data, 
                                            on=['num_vert', 'roi', 'roi_mmp', 'subject', 'hemi', 'trs'], 
                                            suffixes=(suffix1, suffix2))

                # Save the full tsv
                fn_spec_combined = "task-{}_{}_{}_{}_{}_{}".format(
                    prf_task_eyes_names[0].replace('RightEye', '').replace('LeftEye', ''),
                    preproc_prep, filtering, normalization, avg_method, rois_method_format)
                
                output_fn = '{}/{}_{}_prf-css-deriv.tsv'.format(tsv_dir, subject, 
                                                                         fn_spec_combined)
                
                data.to_csv(output_fn, sep='\t', index=False)
                print(f"Saving: {output_fn}")

                # PARAMETER CORRLATIONS AE/RE vs. FE/LE
                # -------------------------------------
                for corr_param in corr_params:
                    
                    for num_roi, roi in enumerate(rois):

                        # get roi
                        df_roi = data.loc[(data.roi == roi)]

                        # Bin only by AE-RE (independent variable / X-axis)
                        df_bin_AR = df_roi.groupby(pd.qcut(df_roi[f'{corr_param}_AE-RE'], 
                                    q=corr_bin_number,
                                    duplicates='drop'))
                        
                        df_bin = pd.DataFrame()
                        df_bin['roi'] = [roi] * len(df_bin_AR)
                        df_bin['num_bins'] = np.arange(len(df_bin_AR))
                        df_bin['n_vertex'] = df_bin_AR.size().values
                        
                        # AE-RE (X-axis)
                        df_bin[f'{corr_param}_AE-RE_median'] = df_bin_AR.apply(lambda x: weighted_nan_median(x[f'{corr_param}_AE-RE'].values, x[f'{rsq2use}_AE-RE'].values)).values
                        df_bin[f'{corr_param}_AE-RE_ci_upper_bound'] = df_bin_AR.apply(lambda x: weighted_nan_percentile(x[f'{corr_param}_AE-RE'].values, x[f'{rsq2use}_AE-RE'].values, 97.5)).values
                        df_bin[f'{corr_param}_AE-RE_ci_lower_bound'] = df_bin_AR.apply(lambda x: weighted_nan_percentile(x[f'{corr_param}_AE-RE'].values, x[f'{rsq2use}_AE-RE'].values, 2.5)).values
                        
                        # FE-LE (Y-axis) - from the SAME AE-RE bins, no separate binning
                        df_bin[f'{corr_param}_FE-LE_median'] = df_bin_AR.apply(lambda x: weighted_nan_median(x[f'{corr_param}_FE-LE'].values, x[f'{rsq2use}_FE-LE'].values)).values
                        df_bin[f'{corr_param}_FE-LE_ci_upper_bound'] = df_bin_AR.apply(lambda x: weighted_nan_percentile(x[f'{corr_param}_FE-LE'].values, x[f'{rsq2use}_FE-LE'].values, 97.5)).values
                        df_bin[f'{corr_param}_FE-LE_ci_lower_bound'] = df_bin_AR.apply(lambda x: weighted_nan_percentile(x[f'{corr_param}_FE-LE'].values, x[f'{rsq2use}_FE-LE'].values, 2.5)).values
                        
                        # Weight by AE-RE R² only (since that's what's being binned)
                        df_bin[f'{rsq2use}_median'] = np.array(df_bin_AR[f'{rsq2use}_AE-RE'].median())
                        
                        # Across roi
                        if num_roi == 0: df_bins = df_bin
                        else: df_bins = pd.concat([df_bins, df_bin]) 

                    tsv_fn = "{}/{}_{}_{}-corr.tsv".format(tsv_dir, subject, fn_spec_combined, corr_param)
                    print('Saving tsv: {}'.format(tsv_fn))
                    df_bins.to_csv(tsv_fn, sep="\t", na_rep='NaN', index=False)

                


In [ ]:
# to add to figures
figure_info=analysis_info
figure_info['corr_params'] = ['prf_rsq', 'prf_x', 'prf_y', 'prf_size', 'prf_ecc', 'pcm_median']

# Axis settings for correlation plots
corr_plot_settings = {'prf_rsq': {'axes': 'pRF R<sup>2',
                                  'range': [0, 0.8],
                                  'tick_step': 0.2,
                                  'n_ticks': 5},
                      'prf_x': {'axes': 'pRF x coord. (dva)',
                                'range': [-5, 5],
                                'tick_step': 2.5,
                                'n_ticks': 5},
                      'prf_y': {'axes': 'pRF y coord. (dva)',
                                'range': [-5, 5],
                                'tick_step': 2.5,
                                'n_ticks': 5},
                      'prf_size': {'axes': 'pRF size (dva)',
                                   'range': [0, 4],
                                   'tick_step': 1,
                                   'n_ticks': 5},
                      'prf_ecc': {'axes': 'pRF ecc. (dva)',
                                  'range': [0, 10],
                                  'tick_step': 2,
                                  'n_ticks': 6},
                      'pcm_median': {'axes': 'pRF CM (mm/dva)',
                                     'range': [0, 30],
                                     'tick_step': 5,
                                     'n_ticks': 7}
                     }

figure_info['rois'] = rois
figure_info['subject_group'] = subject_group

# to make it automatic

In [ ]:
# General imports
import numpy as np

# Figure imports
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Debug
import ipdb
deb = ipdb.set_trace

def plotly_template(template_specs):
    """
    Define the template for plotly
    Parameters
    ----------
    template_specs : dict
        dictionary contain specific figure settings
    
    Returns
    -------
    fig_template : plotly.graph_objs.layout._template.Template
        Template for plotly figure
    """
    import plotly.graph_objects as go
    fig_template=go.layout.Template()

    # Violin plots
    fig_template.data.violin = [go.Violin(
                                    box_visible=False,
                                    points=False,
                                    # opacity=1,
                                    line_color= "rgba(0, 0, 0, 1)",
                                    line_width=template_specs['rois_plot_width'],
                                    width=0.8,
                                    #marker_symbol='x',
                                    #marker_opacity=1,
                                    hoveron='violins',
                                    meanline_visible=False,
                                    # meanline_color="rgba(0, 0, 0, 1)",
                                    # meanline_width=template_specs['rois_plot_width'],
                                    showlegend=False,
                                    )]

    # Barpolar
    fig_template.data.barpolar = [go.Barpolar(
                                    marker_line_color="rgba(0,0,0,1)",
                                    marker_line_width=template_specs['rois_plot_width'], 
                                    showlegend=False, 
                                    )]
    # Pie plots
    fig_template.data.pie = [go.Pie(textposition=["inside","none"],
                                    # marker_line_color=['rgba(0,0,0,1)','rgba(255,255,255,0)'],
                                    marker_line_width=0,#[template_specs['rois_plot_width'],0],
                                    rotation=0,
                                    direction="clockwise",
                                    hole=0.4,
                                    sort=False,
                                    )]

    # Layout
    fig_template.layout = (go.Layout(# general
                                    font_family=template_specs['font'],
                                    font_size=template_specs['axes_font_size'],
                                    plot_bgcolor=template_specs['bg_col'],

                                    # # x axis
                                    xaxis_visible=True,
                                    xaxis_linewidth=template_specs['axes_width'],
                                    xaxis_color= template_specs['axes_color'],
                                    xaxis_showgrid=False,
                                    xaxis_ticks="outside",
                                    xaxis_ticklen=8,
                                    xaxis_tickwidth = template_specs['axes_width'],
                                    xaxis_title_font_family=template_specs['font'],
                                    xaxis_title_font_size=template_specs['title_font_size'],
                                    xaxis_tickfont_family=template_specs['font'],
                                    xaxis_tickfont_size=template_specs['axes_font_size'],
                                    xaxis_zeroline=False,
                                    xaxis_zerolinecolor=template_specs['axes_color'],
                                    xaxis_zerolinewidth=template_specs['axes_width'],
                                    # xaxis_range=[0,1],
                                    xaxis_hoverformat = '.1f',
                                    
                                    # y axis
                                    yaxis_visible=True,
                                    yaxis_linewidth=template_specs['axes_width'],
                                    yaxis_color= template_specs['axes_color'],
                                    yaxis_showgrid=False,
                                    yaxis_ticks="outside",
                                    yaxis_ticklen=8,
                                    yaxis_tickwidth = template_specs['axes_width'],
                                    yaxis_tickfont_family=template_specs['font'],
                                    yaxis_tickfont_size=template_specs['axes_font_size'],
                                    yaxis_title_font_family=template_specs['font'],
                                    yaxis_title_font_size=template_specs['title_font_size'],
                                    yaxis_zeroline=False,
                                    yaxis_zerolinecolor=template_specs['axes_color'],
                                    yaxis_zerolinewidth=template_specs['axes_width'],
                                    yaxis_hoverformat = '.1f',

                                    # bar polar
                                    polar_radialaxis_visible = False,
                                    polar_radialaxis_showticklabels=False,
                                    polar_radialaxis_ticks='',
                                    polar_angularaxis_visible = False,
                                    polar_angularaxis_showticklabels = False,
                                    polar_angularaxis_ticks = ''
                                    ))

    # Annotations
    fig_template.layout.annotationdefaults = go.layout.Annotation(
                                    font_color=template_specs['axes_color'],
                                    font_family=template_specs['font'],
                                    font_size=template_specs['title_font_size'])

    return fig_template


In [ ]:
#def corr_plot(df, figure_info, rsq2use):
"""
Make correlation plot across eyes

Parameters
----------
df : A data dataframe
figure_info : dict with figure settings
rsq2use : rsquare to use

Returns
-------
fig : eccentricy as a function of size plot
"""

#from maths_utils import weighted_regression

# General figure settings
template_specs = dict(axes_color="rgba(0, 0, 0, 1)",
                      axes_width=2,
                      axes_font_size=15,
                      bg_col="rgba(255, 255, 255, 1)",
                      font='Arial',
                      title_font_size=15,
                      rois_plot_width=1.5)

# General figure settings
fig_template = plotly_template(template_specs)
roi_colors = figure_info['roi_colors']
fig_margin = figure_info['rois_fig_margin']
rois = figure_info['rois']
corr_params = figure_info['corr_params']
rows, cols = len(rois), len(corr_params)

rois_hor_spacing = figure_info['rois_hor_spacing']
rois_ver_spacing = figure_info['rois_ver_spacing']
rois_plot_height = figure_info['rois_plot_height']
rois_plot_width = figure_info['rois_plot_width']
subject_group = figure_info['subject_group']

fig_height = rois_plot_height * rows + fig_margin[1] + fig_margin[3] + (rois_ver_spacing * (rows-1))
fig_width = rois_plot_width * cols + fig_margin[0] + fig_margin[2] + (rois_hor_spacing * (cols-1))
hor_spacing = rois_hor_spacing / (fig_width - fig_margin[0] - fig_margin[2])
ver_spacing = rois_ver_spacing / (fig_height - fig_margin[1] - fig_margin[3])

# General settings
fig = make_subplots(rows=rows, cols=cols, print_grid=False, 
                   horizontal_spacing=hor_spacing,
                   vertical_spacing=ver_spacing)


for l, corr_param in enumerate(corr_params):
    tsv_fn = f'/home/mszinte/disks/meso_S/data/amblyo7T_prf/derivatives/pp_data/sub-06/fsnative/prf/tsv/sub-06_task-BarsBarsRingsWedges_fmriprep_dct_z-score_concat_rois-group-mmp_{corr_param}-corr.tsv'
    df = pd.read_table(tsv_fn, sep="\t") 
    
    for j, roi in enumerate(rois):
        
        # Parametring colors
        roi_color = roi_colors[roi]
        axis_settings = corr_plot_settings[corr_param]
        
        # Get data
        df_roi = df.loc[(df.roi == roi)]
            
        # AE-RE
        param_AR_median = np.array(df_roi[f'{corr_param}_AE-RE_median'])
        param_AR_upper_bound = np.array(df_roi[f'{corr_param}_AE-RE_ci_upper_bound'])
        param_AR_lower_bound = np.array(df_roi[f'{corr_param}_AE-RE_ci_lower_bound'])

        # FE-LE
        param_FL_median = np.array(df_roi[f'{corr_param}_FE-LE_median'])
        param_FL_upper_bound = np.array(df_roi[f'{corr_param}_FE-LE_ci_upper_bound'])
        param_FL_lower_bound = np.array(df_roi[f'{corr_param}_FE-LE_ci_lower_bound'])

        # Binned r2 median
        r2_median = np.array(df_roi[f'{rsq2use}_median'])

        # Define line_x
        line_x = np.linspace(axis_settings['range'][0], axis_settings['range'][1], 50)
        
        # Diagonal reference
        fig.add_trace(go.Scatter(x=line_x, y=line_x, mode='lines',
                                  line=dict(color='rgba(0, 0, 0, 1)', width=2, dash='dash'), 
                                  showlegend=False), 
                      row=j+1, col=l+1)

        # Linear regression
        slope, intercept = weighted_regression(param_AR_median, param_FL_median, r2_median, model='linear')
        line = slope * line_x + intercept
        fig.add_trace(go.Scatter(x=line_x, y=line, mode='lines',
                                  line=dict(color=roi_color, width=3), showlegend=False), 
                      row=j+1, col=l+1)

        # Scale marker size by R² (normalized per ROI)
        min_size = 5
        max_size = 15
        r2_min = np.nanmin(r2_median)
        r2_max = np.nanmax(r2_median)
        r2_normalized = (r2_median - r2_min) / (r2_max - r2_min + 1e-8)  # avoid division by zero
        marker_size = min_size + (r2_normalized * (max_size - min_size))
        
        # Markers
        fig.add_trace(go.Scatter(x=param_AR_median, 
                                 y=param_FL_median, mode='markers', 
                                 
                                 error_x=dict(type='data', 
                                              array=param_AR_upper_bound - param_AR_median, 
                                              arrayminus=param_AR_median - param_AR_lower_bound,
                                              visible=True, 
                                              thickness=3, 
                                              width=0, 
                                              color=roi_color),
                                 error_y=dict(type='data', 
                                              array=param_FL_upper_bound - param_FL_median, 
                                              arrayminus=param_FL_median - param_FL_lower_bound,
                                              visible=True, 
                                              thickness=3, 
                                              width=0, 
                                              color=roi_color),
                                 marker=dict(color=roi_color,
                                             symbol='square',
                                             size=marker_size, 
                                             opacity=1.0, 
                                             line=dict(color=roi_color, 
                                                       width=3)), 
                                  showlegend=False), 
                          row=j + 1, 
                          col=l + 1)
        

        # Add legend
        r = axis_settings['range']
        annotation = go.layout.Annotation(x=r[1] - 0.05 * (r[1] - r[0]), 
                                          y=r[1] - 0.9 * (r[1] - r[0]),
                                          text=roi, xanchor='right',
                                          showarrow=False, font_color=roi_color, 
                                          font_family=template_specs['font'],
                                          font_size=template_specs['axes_font_size'])
        fig.add_annotation(annotation, row=j + 1, col=l+1)

        # Set axis
        if j == len(rois) - 1:  # Last row (bottom)
            fig.update_xaxes(title_text=f"AE – {axis_settings['axes']}", 
                             range=axis_settings['range'],
                             tickmode='linear',
                             tick0=axis_settings['range'][0],
                             dtick=axis_settings['tick_step'],
                             showline=True, 
                             row=j + 1, col=l + 1)
        else:
            fig.update_xaxes(title_text='', 
                             range=axis_settings['range'],
                             tickmode='linear',
                             tick0=axis_settings['range'][0],
                             dtick=axis_settings['tick_step'],
                             showline=True, 
                             row=j + 1, col=l + 1)
        
        fig.update_yaxes(title_text=f"FE – {axis_settings['axes']}", 
                         range=axis_settings['range'],
                         tickmode='linear',
                         tick0=axis_settings['range'][0],
                         dtick=axis_settings['tick_step'],
                         showline=True, 
                         row=j + 1, col=l + 1)

        
fig.update_layout(height=fig_height, 
                  width=fig_width, 
                  showlegend=False, 
                  template=fig_template,
                  margin_l=fig_margin[0], 
                  margin_t=fig_margin[1], 
                  margin_r=fig_margin[2], 
                  margin_b=fig_margin[3]
                 )
fig.show()
#return fig